# Video to synthetic RAW - video v1 (single run)

One **Runtime > Run all** turns one video clip into synthetic Sony A1 II style RAW with the PRD v6 pipeline (inverse
ISP to a linear RGGB mosaic, A1 II shot and read noise). Every output carries the same mosaic, at the source resolution
(1080p stays 1920 x 1080):

| Output | Format |
|---|---|
| one MLV per clip | Magic Lantern raw video, 14-bit RGGB, the clip frame rate; MLV App opens it and exports ProRes or CinemaDNG |
| one TIFF per frame | developed 16-bit RGB, sRGB primaries and transfer, embedded sRGB ICC profile |
| one DNG per frame (`WRITE_DNG = True`, off by default) | 16-bit RGGB Bayer mosaic, CinemaDNG-style names with frame rate and time code |
| preview | MP4 of the developed TIFFs beside the source |

Illuminant (the source's own D65 white), white balance and exposure are fixed for the whole clip; every frame gets its
own seeded noise. Every TIFF must read back, every DNG (when written) must open in LibRaw and pass Adobe dng_validate, and the MLV is
read back and decoded bit-exact. The verdict is COMPLETE, FLAGGED (it says which frames or which check)
or INVALID (a step failed; it says why). There is **no accuracy claim**: no real RAW of the clip exists, and the v6
95% benchmark does not transfer to video frames.

## Before you press Run all

1. **Runtime > Change runtime type: A100 GPU (High-RAM) or L4 GPU.** The conversion runs on the CPU; these runtimes
   bring more CPU cores, RAM and disk. A CPU-only runtime also runs, with a warning, and is slower.
2. With `VIDEO = None` (the default) the driver cell shows a **Choose Files** button: pick one clip, for example a
   1080p MP4 of about 5 seconds. Or set `VIDEO` to a direct URL, a Google Drive share link of a file shared as
   "Anyone with the link", or a path on the runtime such as `/content/drive/MyDrive/clip.mp4` (Drive is mounted for
   you, one prompt). Audio is dropped: the outputs have none.
3. A 1080p clip of about 5 seconds should finish well under an hour, setup included. Keep the tab open. The report
   records the measured seconds per frame and estimates what a 4K version would cost.

## What arrives

When the run ends (also after a failure) the driver zips the results and downloads them: first
`<run_id>_01_summary.zip` (report.html with the side-by-side preview, the stability check, per-frame timings, the
manifests and the verdict), then the MLV clip (`<run_id>_NN_mlv_partKofN.zip`), the TIFFs (`_NN_tiff_...`) and, with `WRITE_DNG = True`, the DNGs
(`_NN_dng_...`), NN numbered on continuously from 02, at most 500 MB each.
Chrome may ask to allow multiple downloads for colab.research.google.com: allow it. If a download was blocked, the tab
was closed or the kernel crashed, the driver cell prints a one-line recovery command at the start; paste it into a new
cell to rebuild and download every archive. Files on the runtime are lost when Colab recycles it unless
`DRIVE_MIRROR = True`.

In [ ]:
# Settings. Run all needs no edits: VIDEO = None shows an upload button for the clip.
REPO_REF = "bf1b7f8cd0e8362c4df5f39d673239572cd4f041"  # pinned commit of the code this notebook runs
# VIDEO: None (upload button), a direct URL, a Google Drive share link ("Anyone with the link") or a runtime path
VIDEO = None
DRIVE_MIRROR = False  # True mounts Google Drive first (one prompt) and mirrors the summary after every step
WRITE_DNG = False  # True adds the DNG sequence (with its LibRaw and dng_validate checks) beside the MLV and TIFFs

## Driver

Runs steps 1-5 (input, setup, preflight, convert, develop and check) inside one `try/finally`, so packaging and
downloads happen after success and after any failure. No edits needed.

In [ ]:
# Driver: steps 1-5 inside one try/finally; the finally block packages and downloads (video_v1/README.md).
import datetime
import hashlib
import json
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as error:
    raise RuntimeError("This notebook runs only in Google Colab (video_v1/README.md)") from error

SETTINGS = {
    "REPO_REF": REPO_REF,
    "VIDEO": VIDEO,
    "DRIVE_MIRROR": DRIVE_MIRROR,
    "WRITE_DNG": WRITE_DNG,
}
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"
CLONE = Path("/content/experiment_50MP")
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + REPO_REF[:7]
RUN_DIR = Path("/content/exp50mp_video_v1") / RUN_ID
RUN_DIR.mkdir(parents=True)
CONFIG = {"run_id": RUN_ID, "clone": str(CLONE), "settings": SETTINGS}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(f"run_dir: {RUN_DIR}")
print(f"clone: {CLONE}")
print("If the downloads are blocked or the kernel crashes, run this in a new cell:")
print(
    f'import sys; sys.path.insert(0, "{CLONE}"); from video_v1.delivery import download_results; '
    f'download_results("{RUN_DIR}")',
    flush=True,
)


def _now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _mark(step, name, *, started=False, error=None):
    """Steps 1 and 2 run before the package exists; same state.json format as video_v1.delivery."""
    path = RUN_DIR / "state.json"
    state = json.loads(path.read_text()) if path.exists() else {"steps": []}
    record = next((r for r in state["steps"] if r["step"] == step), None)
    if record is None:
        record = {"step": step, "name": name}
        state["steps"].append(record)
    if started:
        state["step_started"] = step
        record.update(status="started", started_at=_now())
    else:
        record.update(status="failed" if error else "finished", finished_at=_now())
        if error is not None:
            record["error"] = f"{type(error).__name__}: {error}"
            record["traceback"] = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    path.write_text(json.dumps(state, indent=2))
    line = f"{_now()} step {step} {record['status']}: {name}"
    print(line, flush=True)
    with open(RUN_DIR / "run.log", "a") as handle:
        handle.write(line + "\n")


def _input():
    """Step 1 runs first, while you are at the tab: the Drive prompt, then the upload button when VIDEO is None."""
    if DRIVE_MIRROR or str(VIDEO or "").startswith("/content/drive/"):
        from google.colab import drive

        drive.mount("/content/drive")
    if VIDEO is not None:
        return
    from google.colab import files

    print("Choose one video file (the upload shows its progress below the button):", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"upload exactly one video file (got {len(uploaded)}); or set VIDEO and run all again")
    ((name, data),) = uploaded.items()
    safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in Path(name).name).strip("._") or "source_video"
    path = RUN_DIR / "inputs" / safe
    path.parent.mkdir(exist_ok=True)
    path.write_bytes(data)
    record = {"path": str(path), "name": name, "bytes": len(data)}
    (RUN_DIR / "inputs" / "upload.json").write_text(json.dumps(record, indent=2))
    print(f"uploaded {name}: {len(data) / 1e6:.1f} MB", flush=True)


def _setup():
    git = ["git", "-C", str(CLONE)]
    if not (CLONE / ".git").exists():
        subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(CLONE)], check=True)
    subprocess.run(git + ["fetch", "origin", REPO_REF], check=True)
    subprocess.run(git + ["checkout", "--detach", REPO_REF], check=True)
    head = subprocess.check_output(git + ["rev-parse", "HEAD"], text=True).strip()
    if head != REPO_REF:
        raise RuntimeError(f"checked out {head}, expected REPO_REF {REPO_REF}")
    packages = {"video_v1", "prd_v6", "prd_v5", "synthraw", "synthetic_raw_lab"}
    for name in [m for m in sys.modules if m.split(".")[0] in packages]:
        del sys.modules[name]
    sys.path.insert(0, str(CLONE))
    from prd_v6 import install

    install.run(RUN_DIR, CLONE)


def _fallback(reason):
    """Minimal packaging when the clone or its delivery module is unavailable (standard library only)."""
    archive = RUN_DIR.parent / f"{RUN_ID}_fallback.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RUN_DIR.rglob("*")):
            if path.is_file() and path.suffix.lower() in {".json", ".log", ".txt", ".csv", ".html"}:
                bundle.write(path, path.relative_to(RUN_DIR.parent))
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    print(f"download {archive.name}  {archive.stat().st_size / 1e6:.1f} MB  sha256 {digest}", flush=True)
    try:
        from google.colab import files

        files.download(str(archive))
    except Exception as error:  # noqa: BLE001
        print(f"download failed ({type(error).__name__}: {error}); use the Files pane: {archive}")
    rule = "=" * 78
    print(f"{rule}\nVIDEO v1 RUN {RUN_ID}: INVALID\nINVALID: {reason}\nRun folder: {RUN_DIR}\n{rule}", flush=True)


FAILURE = None
try:
    for step, name, task in ((1, "Input", _input), (2, "Setup", _setup)):
        _mark(step, name, started=True)
        try:
            task()
        except BaseException as error:
            _mark(step, name, error=error)
            raise
        _mark(step, name)
    from video_v1 import delivery, driver

    delivery.checkpoint(RUN_DIR, SETTINGS)
    driver.run(RUN_DIR, SETTINGS)
except BaseException as error:
    state = json.loads((RUN_DIR / "state.json").read_text()) if (RUN_DIR / "state.json").exists() else {}
    FAILURE = {"step": state.get("step_started"), "error": f"{type(error).__name__}: {error}"}
    traceback.print_exc()
finally:
    try:
        from video_v1.delivery import finalize_run

        VERDICT, ARCHIVES = finalize_run(RUN_DIR, SETTINGS, FAILURE)
    except Exception as error:  # noqa: BLE001 - the fallback must still deliver something
        traceback.print_exc()
        _fallback(f"{FAILURE['error'] if FAILURE else 'packaging failed'}; packaging: {type(error).__name__}: {error}")